# Creación del Pipeline

In [1]:
# ==================== Importaciones ====================
import os
import pandas as pd
import joblib
import re
import unicodedata
import numpy as np
from num2words import num2words
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk import word_tokenize
import nltk

# ==================== Descargar recursos NLTK ====================
nltk.download('stopwords')
nltk.download('punkt')

# ==================== Configuración de stopwords y stemmer ====================
stop_words = stopwords.words('spanish')
stemmer = SnowballStemmer("spanish")

# ==================== Funciones de Preprocesamiento ====================
def remove_non_ascii(words):
    """Eliminar caracteres no ASCII"""
    return [unicodedata.normalize('NFKD', w).encode('ascii', 'ignore').decode('utf-8', 'ignore') for w in words if w]

def normalize_text(text):
    """Normalizar texto eliminando acentos y signos de puntuación"""
    text = text.replace('á', 'a').replace('é', 'e').replace('í', 'i')
    text = text.replace('ó', 'o').replace('ú', 'u').replace('ü', 'u')
    text = text.replace('ñ', 'n')
    text = re.sub(r'[^\w\s]', '', text)
    return text.lower()

def to_lowercase(words):
    """Convertir palabras a minúsculas"""
    return [w.lower() for w in words]

def remove_punctuation(words):
    """Eliminar signos de puntuación de cada palabra"""
    return [re.sub(r'[^\w\s]', '', w) for w in words if w]

def replace_numbers(words):
    """Reemplazar números con su representación textual en español"""
    return [num2words(w, lang='es') if w.isdigit() else w for w in words]

def remove_stopwords(words):
    """Eliminar stopwords en español"""
    return [w for w in words if w not in stop_words]

def stem_words(words):
    """Aplicar stemming a las palabras"""
    return [stemmer.stem(w) for w in words]

def preprocessing(text):
    """Pipeline completo de preprocesamiento de texto:
    Tokeniza, elimina puntuación, pasa a minúsculas, reemplaza números, 
    elimina caracteres no ASCII, quita stopwords y aplica stemming."""
    words = word_tokenize(text)
    words = remove_punctuation(words)
    words = to_lowercase(words)
    words = replace_numbers(words)
    words = remove_non_ascii(words)
    words = remove_stopwords(words)
    words = stem_words(words)
    return ' '.join(words)

def transfor_data_local(df):
    """Aplicar preprocesamiento a columnas 'Titulo' y 'Descripcion' y crear columna 'Texto_Procesado'"""
    columnas = ['Titulo', 'Descripcion']
    # Asegurarse de que las columnas existan y llenar valores nulos
    df = df.copy()
    for col in columnas:
        if col not in df.columns:
            raise ValueError(f"La columna {col} no existe en el DataFrame.")
    df = df[columnas].fillna('')
    # Aplicar preprocesamiento
    for columna in columnas:
        df[columna] = df[columna].apply(preprocessing)
    # Concatenar ambas columnas en un solo texto
    df['Texto_Procesado'] = df['Titulo'] + ' ' + df['Descripcion']
    return df['Texto_Procesado']

# ==================== Función para Crear y Entrenar el Modelo ====================
def createModel(text_transformer, df):
    """Crear y entrenar el modelo usando un pipeline.
    Se preprocesa el texto, se vectoriza con el transformador definido y se entrena un RandomForest."""
    # Asegurarse de que las columnas requeridas existan
    for col in ['Titulo', 'Descripcion', 'Label']:
        if col not in df.columns:
            raise ValueError(f"La columna {col} es requerida en el DataFrame.")
            
    # Eliminar filas con valores nulos en las columnas críticas
    df = df.dropna(subset=['Titulo', 'Descripcion', 'Label'])
    X = df[['Titulo', 'Descripcion']]
    y = df['Label']
    
    # Transformador para aplicar el preprocesamiento sobre el DataFrame
    data_transformer = FunctionTransformer(transfor_data_local)
    
    # Separar datos en entrenamiento y validación
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )
    
    # Modelo base: RandomForest
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        random_state=42
    )
    
    # Definición del pipeline
    pipeline = Pipeline([
        ("data_transform", data_transformer),
        ("vectorizer", text_transformer),
        ("classifier", model)
    ])
    
    # Entrenar el modelo
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Métricas de evaluación
    results = classification_report(y_test, y_pred, output_dict=True)
    print("Reporte de Clasificación:\n", classification_report(y_test, y_pred))
    print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred))
    
    return results, pipeline

# ==================== Función para Cargar Modelo ====================
def loadModel(MODEL_PATH):
    """Cargar un modelo entrenado desde un archivo .joblib"""
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Modelo no encontrado en {MODEL_PATH}")
    return joblib.load(MODEL_PATH)

# ==================== Cargar Datos y Entrenar Modelo ====================
# Cargar datos desde el archivo CSV
try:
    data = pd.read_csv("./../data/fake_news_spanish.csv", sep=";")
except Exception as e:
    raise FileNotFoundError("No se pudo cargar el archivo CSV. Verifica la ruta y el formato.") from e

# Eliminar columna 'ID' si existe, duplicados y valores nulos
data = data.drop(columns=['ID'], errors='ignore')
data = data.drop_duplicates().dropna()

# Crear modelo usando TfidfVectorizer (se pueden ajustar parámetros según el dataset)
text_transformer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
results, modelo_final = createModel(text_transformer, data)

# Guardar modelo entrenado en un archivo .joblib
joblib.dump(modelo_final, "../models/Predictor.joblib")
print("Modelo guardado como 'Predictor.joblib'.")

# ==================== Cargar Modelo y Predecir Nuevas Noticias ====================
# Cargar modelo para realizar predicciones
modelo_cargado = loadModel("../models/Predictor.joblib")

# Nuevas noticias para predecir
nuevas_noticias = pd.DataFrame({
    "Titulo": [
        "El Gobierno aprueba una nueva ley de educación",
        "Detienen a un político corrupto en España"
    ],
    "Descripcion": [
        "La ley busca mejorar el acceso a la educación pública.",
        "El político fue arrestado tras ser descubierto en un caso de corrupción."
    ]
})

# Realizar predicciones sobre las nuevas noticias
predicciones = modelo_cargado.predict(nuevas_noticias)

# Mostrar resultados de las predicciones
print("\n===== Resultados de Predicción =====")
for i, pred in enumerate(predicciones):
    etiqueta = 'Real' if pred == 1 else 'Falsa'
    print(f"Noticia {i+1}: {etiqueta}")

# Guardar predicciones en un archivo CSV
nuevas_noticias["Prediccion"] = predicciones
nuevas_noticias.to_csv("predicciones_noticias.csv", sep=";", index=False)
print("Predicciones guardadas en 'predicciones_noticias.csv'")


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/trodriten/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/trodriten/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.96      0.88      0.92      7185
           1       0.92      0.97      0.95      9796

    accuracy                           0.94     16981
   macro avg       0.94      0.93      0.93     16981
weighted avg       0.94      0.94      0.94     16981

Matriz de Confusión:
 [[6352  833]
 [ 247 9549]]
Modelo guardado como 'Predictor.joblib'.

===== Resultados de Predicción =====
Noticia 1: Real
Noticia 2: Real
Predicciones guardadas en 'predicciones_noticias.csv'
